# 05 · SCAN-Lite compositional mapping (v17)

Walk through the actual v17 synthetic data construction: five commands, uppercase action mapping, repetitions 1–5, shuffled 25 pairs, then an 80/20 row split. This deliberately simple repeat grammar is **not full SCAN**. The supplied `DIINA_Benchmark_datasets_scan_loader.py` is empty.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
src= (PROJECT/"DIINA_v17_REAL_BENCHMARK.py").read_text()
print("v17 source contains commands/repetition loop:", all(t in src for t in ["commands =", "for rep in range(1,6)", "random.shuffle(pairs)"]))
commands=["jump","walk","run","look","turn"]
actions=[x.upper() for x in commands]
pairs=[{"input":[c]*r,"output":[a]*r,"repeat":r} for c,a in zip(commands,actions) for r in range(1,6)]
print("Count:",len(pairs),"examples; preview:",pairs[:3]); assert len(pairs)==25
try:
    import ipywidgets as widgets
    @widgets.interact(command=commands, repetition=(1,5,1))
    def show_mapping(command="jump",repetition=1):
        action=actions[commands.index(command)]; print(" ".join([command]*repetition),"→"," ".join([action]*repetition))
except ImportError:
    print("ipywidgets unavailable; use mapping(command, repetition) below")
    def show_mapping(command="jump",repetition=1): return (" ".join([command]*repetition)," ".join([command.upper()]*repetition))

In [ ]:
def mapping(command,repetition):
    if command not in commands or repetition not in range(1,6): raise ValueError("Use a known command and repetition 1..5")
    return " ".join([command]*repetition)," ".join([command.upper()]*repetition)
print(mapping("turn",4))
loader=(PROJECT/"DIINA_Benchmark_datasets_scan_loader.py").read_text(); print("Loader byte/character count:",len(loader)); print("v17 states 80% split; shuffled with Python random seed 42.")
assert mapping("jump",2)==("jump jump","JUMP JUMP")

**Scope:** The R counterpart is a deterministic set/grammar port. Neither it nor this notebook trains the PyTorch networks. Python v17 uses a Transformer and a GRU-backed gated DIINA baseline, not the manuscript's two dedicated streams.